In [22]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [28]:
#transforms.Compose()

#Think of it as:
    #Image
    # ↓
    #Resize
    # ↓
    #ToTensor
    # ↓
    #Ready for CNN

#It applies multiple transformations one after another.
#-----------------------------------------------------#
#transforms.Resize((128,128))
#MRI images may have different sizes:

#200×200
#512×512
#300×400
#CNNs need a fixed size.
#So we resize everything to:
#128×128

#-----------------------------------------------------#
#transforms.ToTensor()
#Images are normally:

#JPEG / PNG files

#CNNs can't work directly with image files.

#ToTensor() converts them into:

transform= transforms.Compose([transforms.Resize((128,128)),
                               transforms.ToTensor()
                            ])

In [29]:
# Load Dataset
dataset= datasets.ImageFolder(
     root=r'C:\Users\telji\OneDrive\Desktop\Brain Tumor Detection using CNN\archive\brain_tumor_dataset',
     transform=transform
)
print(len(dataset))
print(dataset.classes)
print(dataset.class_to_idx)


253
['no', 'yes']
{'no': 0, 'yes': 1}


In [30]:
#Splitting data
from torch.utils.data import random_split
train_size=int(0.8*len(dataset))
test_size=len(dataset)-train_size

train_data,test_data= random_split(dataset,
                                   [train_size,test_size]
                                   )

# creating dataloaders
tranning_loader=DataLoader(
    train_data,
    batch_size=32,
    shuffle=True
)

testing_loader=DataLoader(
    test_data,
    batch_size=32,
    shuffle=False
)
print("Total Images:", len(dataset))
print("Training Images:", len(train_data))
print("Testing Images:", len(test_data))

Total Images: 253
Training Images: 202
Testing Images: 51


In [31]:
# CNN MODEL

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Convolution Layers
        self.conv1=nn.Conv2d(in_channels=3,
                             out_channels=16,
                             kernel_size=3
                             )                      
        self.conv2=nn.Conv2d(in_channels=16, 
                             out_channels=32, 
                             kernel_size=3
                             )                      
        #Activation Layer
        self.relu=nn.ReLU()
        #pooling
        self.pool=nn.MaxPool2d(2)
        #flatten
        self.flatten = nn.Flatten()
        # Fully Connected Layers
        self.fc1 = nn.Linear(28800, 128)  #2880 features, 128 neurons 
        self.fc2 = nn.Linear(128, 2)     #128 features  from fc1 neurons, 2 neurons(1 neuron for each lable)
    def forward(self, x):

        x = self.conv1(x) #------>128-3+2(0)/1 + 1 = 126
        x = self.relu(x)
        x = self.pool(x)  #------> 126/2= 63 (half of 126)

        x = self.conv2(x) #------> 63-3+2(0)/1 + 1 = 61
        x = self.relu(x)
        x = self.pool(x)  #------> 61/2 = 30.5

        x = self.flatten(x) #-----> 32(batch size)x 32(conv2 outchannels)x 30 x 30 =28800
        x = self.fc1(x)
        x = self.relu(x)

        x = self.fc2(x)

        return x

In [33]:
#sample testing
model= CNN()
images, labels = next(iter(tranning_loader))

output = model(images)

print(output.shape)

#32 images
#2 logits for each image

#Neuron 0 → No Tumor
#Neuron 1 → Tumor

torch.Size([32, 2])


In [34]:
loss_fn= nn.CrossEntropyLoss()
optimiser=optim.Adam(model.parameters(), lr=0.001)


In [37]:
for epoch in range (20):
    for images, labels in tranning_loader:
        output=model(images)
        loss= loss_fn(output,labels)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.0573
Epoch 2, Loss: 0.0209
Epoch 3, Loss: 0.0293
Epoch 4, Loss: 0.0095
Epoch 5, Loss: 0.0016
Epoch 6, Loss: 0.0005
Epoch 7, Loss: 0.0030
Epoch 8, Loss: 0.0038
Epoch 9, Loss: 0.0025
Epoch 10, Loss: 0.0021
Epoch 11, Loss: 0.0042
Epoch 12, Loss: 0.0039
Epoch 13, Loss: 0.0042
Epoch 14, Loss: 0.0007
Epoch 15, Loss: 0.0007
Epoch 16, Loss: 0.0011
Epoch 17, Loss: 0.0010
Epoch 18, Loss: 0.0012
Epoch 19, Loss: 0.0024
Epoch 20, Loss: 0.0001


In [41]:
correct = 0
total = len(test_data)

with torch.no_grad():

    for images, labels in testing_loader:

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        correct += (predicted == labels).sum().item()

accuracy = (correct / total) * 100

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 92.16%


In [42]:
print(model)

CNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=28800, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=2, bias=True)
)


In [43]:
torch.save(
    model.state_dict(),
    "brain_tumor_cnn.pth"
)